In [9]:
import pandas as pd
import os
import torch
import numpy as np
from eval import score_heatmap

# 1. Initial Configurations
csv_path = r'C:\Repositories\picard-anomalydetection\data\BCS-DBT-boxes-validation-v2-PHASE-2-Jan-2024.csv'
df_boxes = pd.read_csv(csv_path)

# Main 'data' folder path (which now contains the patient subfolders)
heatmaps_folder = r'C:\Repositories\picard-anomalydetection\heatmaps\Breast-Cancer-Screening-DBT\dropout\03-08-2026_17-56-09\data'

# --- THE BIG CHANGE IS HERE ---
# Search for all .pt files by navigating through all subfolders
all_generated_paths = []
for root, dirs, files in os.walk(heatmaps_folder):
    for file in files:
        if file.endswith('.pt'):
            # Store the full file path (e.g.: data/DBT-P001/DBT-P001_...pt)
            all_generated_paths.append(os.path.join(root, file))
# -----------------------------------

auc_results = []
print(f"Starting evaluation based on the {len(df_boxes)} bounding boxes from the CSV...\n")

# 2. Loop through each row of your Bounding Boxes CSV
for index, row in df_boxes.iterrows():
    patient_id = str(row['PatientID'])
    view = str(row['View']).lower()
    slice_num = int(row['Slice'])
    
    # 3. Radar: Find the heatmap corresponding to this row
    found_heatmap_path = None
    for full_path in all_generated_paths:
        # Get only the final filename to perform the check
        filename = os.path.basename(full_path).lower()
        
        # Check if Patient, View, and Slice match
        if patient_id.lower() in filename and view in filename:
            if f"slice{slice_num:03d}" in filename:
                found_heatmap_path = full_path
                break # Found it, can stop searching
                
    # 4. Evaluation
    if found_heatmap_path:
        # Extract the exact coordinates from your CSV
        x = int(row['X'])
        y = int(row['Y'])
        width = int(row['Width'])
        height = int(row['Height'])
        
        ground_truth_box = [(y, x, height, width)]
        
        # Load the heatmap using the full path we found in the search
        heatmap_tensor = torch.load(found_heatmap_path)
        
        # Calculate the score (Area Under Curve)
        perfect_mask, auc_score = score_heatmap(
            score_type='pixel_AUC', 
            heatmap=heatmap_tensor, 
            bboxes=ground_truth_box
        )
        
        lesion_type = row['Class']
        print(f"{patient_id} | {view} | Slice {slice_num} ({lesion_type}) -> AUROC: {auc_score:.4f}")
        
        auc_results.append(auc_score)

# 5. Final Result
print("-" * 70)
if auc_results:
    mean_auc = np.mean(auc_results)
    print(f"MEAN (AUROC): {mean_auc:.4f}")
    print(f"Total lesions evaluated: {len(auc_results)}")
else:
    print("No corresponding heatmaps were found in the subfolders.")

Starting evaluation based on the 75 bounding boxes from the CSV...

DBT-P00114 | rmlo | Slice 25 (cancer) -> AUROC: 0.0958
DBT-P00431 | rmlo | Slice 27 (cancer) -> AUROC: 0.2659
----------------------------------------------------------------------
MEAN (AUROC): 0.1809
Total lesions evaluated: 2
